# **Chess Win Predictor**

## **Abstract**

Chess engines evaluate positions based on perfect play, but real online games are decided by human factors: rating gaps, time pressure, and practical mistakes. This project builds a win-probability predictor using those features (player Elo, remaining clock time, move-time patterns, and a compact board summary) extracted from the Lichess Open Database. The raw monthly export (84.6 million games) is filtered down to 8.6 million completed 10-minute rapid games, which form the basis for all experiments. The pipeline tests logistic regression, gradient-boosted trees, calibrated SVMs, MLPs, and recurrent neural networks, evaluated at 5-move landmarks rather than as a single pooled average. The best result comes from a PyTorch SimpleRNN trained on all 8.6 million games, reaching `63.6%` test accuracy and an average landmark accuracy of `74.2%`. Richer board-state features improved results more than any change in model architecture.

## **Introduction**

The standard approach to chess outcome estimation is engine evaluation: given any board position, a program like Stockfish can say who is theoretically better. But engine evaluation answers a different question from the one this project is interested in. A player can be objectively losing on the board and still win on time; a much lower-rated opponent can outplay a stronger one simply because the stronger player is running out of time. These human factors (Elo gap, remaining clock, time usage across moves) are what this project uses to predict win probability.

The data comes from the Lichess Open Database, which provides monthly PGN exports with clock annotations embedded after every move. Two design constraints matter throughout. First, data leakage: the model must only ever see information available at the moment of prediction, no future moves, no final move counts, and no outcome tags from the PGN header. Second, data quality: the raw export mixes game speeds and includes disconnections and abandoned games, so filtering is a required first step before any feature extraction or modeling.

## **Methodology**

The pipeline has four stages, each building on the last.

### Stage 1: Data filtering

The raw PGN export is streamed game-by-game and two conditions must both be met for a game to be kept:

1. `TimeControl == "600+0"` (standard 10-minute rapid game, no increment)
2. `Termination == "Normal"` (ended by checkmate or resignation, not by disconnection or abandonment)

Streaming rather than loading the file into memory makes the filter practical on the 27.7 GB compressed source.

### Stage 2: Feature extraction

Two feature sets were developed, in increasing order of richness.

**Light features** use only what is directly readable from the PGN metadata: player Elo, remaining clock for both players, time spent on the last move, clock ratios and differences, and simple move-shape signals from the SAN notation (capture, check, castle, promotion). No board reconstruction is needed.

**Board-aware features** add a compact snapshot of the actual chess position after each move. The library `python-chess` replays the game move-by-move from the starting position and, once the board is updated, reports material counts, material imbalance, number of legal moves, castling rights, bishop pair status, and the halfmove clock. This gives the model real chess context (for example, knowing whether a side is up a piece or has lost castling rights) without requiring any engine evaluation.

In both cases the snapshot is taken *immediately after* a move is played, using only information that exists at that instant. No future move or outcome information is ever used.

### Stage 3: Training and data management

At small scale (up to 10k games), features are exported to a flat CSV and models are trained on tabular data using scikit-learn. At larger scale the pipeline switches to **NPZ shards**: each game's full board-aware sequence is written to a compressed shard file, with the train/validation/test split enforced at the game level during extraction. This avoids materializing huge intermediate CSVs and lets the PyTorch trainer stream shards with persistent workers, pinned memory, and mixed precision, with epoch checkpoints so long runs can be paused and resumed.

### Stage 4: Evaluation

All splits are **game-level**: every move snapshot from a given game always lands in the same split, preventing leakage across move rows. Hyperparameters were tuned on the validation set; the test set was only touched for final reported numbers.

The main evaluation design is **landmark evaluation**: instead of averaging a metric over every move snapshot (which would overweight long games), one snapshot per game is taken every 5 full moves. This produces a per-stage accuracy and Brier score curve, showing how confident and well-calibrated each model is in the opening, middlegame, and endgame separately.

## **Experimental Setup**

### Dataset

| Stage | Count |
|---|---|
| Raw games in monthly export | 84,600,043 |
| After filtering (`600+0`, `Termination=Normal`) | 8,633,881 |
| Train / Validation / Test (full dataset) | 6,214,945 / 690,252 / 1,726,823 |

Smaller subsets of `1k`, `10k`, `50k`, `100k`, `250k`, and `1M` games were used for iterative development before the final full-dataset run.

### Running the pipeline

Filtering:

```python
!python scripts/filter_lichess_pgn.py \
    --input lichess_db_standard_rated_2026-02/lichess_db_standard_rated_2026-02.pgn \
    --output data/lichess_rapid_10_0_completed.pgn \
    --time-control 600+0 --termination Normal
```

Board-aware feature extraction (CSV, for small-scale runs):

```python
!python scripts/extract_lichess_board_features.py \
    --input data/lichess_rapid_10_0_completed.pgn \
    --output data/dev_board_features_1000_games.csv --max-games 1000
```

Shard extraction (for large-scale RNN training):

```python
!python scripts/extract_rnn_game_shards_parallel.py \
    --input data/lichess_rapid_10_0_completed.pgn \
    --output-dir data/rnn_shards_250000 --max-games 250000 --workers 2
```

### Models compared

| Family | Variants tested |
|---|---|
| **Light logistic regression** | Logistic regression on Elo + clock + SAN-shape features |
| **Light gradient boosting** | Histogram gradient boosting on the same light feature set |
| **Board logistic regression** | Logistic regression on the full board-aware feature set |
| **Board gradient boosting / XGBoost** | Histogram gradient boosting and XGBoost on board-aware features |
| **Board SVM** | `LinearSVC` with sigmoid probability calibration |
| **Board MLP** | 2–3 hidden layer configurations; one draw-oversampled variant |
| **Keras RNN sweep** | `SimpleRNN(64)`, `GRU(64)`, `LSTM(64)`, stacked GRU, dropout and loss-weighting variants, all on the `1k` development set |
| **PyTorch SimpleRNN** | `RNN(input=41, hidden=64)` from NPZ shards with CUDA mixed precision; scaled from `1k` to all `8.6M` games |

All recurrent models are trained on **game prefix sequences**: each training example is the board-aware feature sequence from move 1 up to a 5-move landmark, and the label is the final game outcome (White win / Black win / Draw).

## **How to Use This Notebook in Google Colab**

There are two ways to run this notebook.

### Lightweight run (for graders)

The fast path. Clones the repo, loads archived results from `artifacts/reported_results/`, and optionally runs a small end-to-end smoke test on a 60-game sample. No large files needed.

**Steps:**

1. **Run Cell 6 (Setup)**: clones the repo and installs dependencies.
2. **Run Cell 7 (Load results)**: prints the SimpleRNN scaling table and the 250k PyTorch RNN reported metrics from the tracked artifact files.
3. **Run Cell 8 (Smoke test, optional)**: runs the full pipeline (filter, extract, shard, train) on a tiny 60-game sample in a few minutes on a standard Colab CPU runtime.

All cells can be run top-to-bottom with **Runtime > Run all**.



### Full reproduction (requires large data files)

> **Important context:** The main experimental work for this project was done **outside of Google Colab**, on a local Windows machine with an NVIDIA RTX 3060 GPU. The raw Lichess PGN for February 2026 is **27.7 GB compressed** and contains **84.6 million games**. After filtering to completed 10-minute rapid games, the result is still **8.6 million games**, which is not feasible on a free Colab runtime.

Full reproduction requires either uploading the pre-filtered `lichess_rapid_10_0_completed.pgn` to Colab or Google Drive, or starting from the raw Lichess PGN (even larger), then running the extraction and training scripts documented in the Experimental Setup section. The archived results in `artifacts/reported_results/` were generated from those full runs and represent the numbers reported in the tables below.

The notebook is the readable report; the external scripts in `scripts/` contain the modular pipeline code. Links to each key script are provided inline in the Methodology section.

### Tracked reproducibility assets

Small files tracked in Git (safe to commit):

- `artifacts/sample_data/colab_dev_60_games.pgn`
- `artifacts/sample_data/sample_rapid_10_0_completed.pgn`
- `artifacts/reported_results/dev_board_logreg_metrics_1000_games.json`
- `artifacts/reported_results/torch_rnn_landmark_board_250000_with_history.json`
- `artifacts/reported_results/simplernn_scaling_summary.csv`

These are separate from `data/`, which stays git-ignored to avoid committing multi-gigabyte intermediate files.

In [ ]:
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/thermoNuke1/ml-PROJECT.git"
REPO_NAME = "ml-PROJECT"
REPO_PATH = Path(f"/content/{REPO_NAME}")

os.chdir("/content")

if not REPO_PATH.is_dir():
    !git clone {REPO_URL} {REPO_PATH}
else:
    print("Already cloned - pulling latest changes")
    !git -C {REPO_PATH} pull

requirements_path = REPO_PATH / "requirements.txt"
if requirements_path.exists():
    !pip install -q -r {requirements_path}
else:
    print("No requirements.txt found - using the current notebook environment")

repo_path_str = str(REPO_PATH)
if repo_path_str not in sys.path:
    sys.path.insert(0, repo_path_str)

os.chdir(REPO_PATH)
print(f"Working directory: {os.getcwd()}")
print(f"Python path includes: {repo_path_str}")


In [1]:
from pathlib import Path
import json
import pandas as pd

# archived results live in artifacts/reported_results/; data/ is git-ignored
artifacts_dir = Path('artifacts/reported_results')

scaling_path = artifacts_dir / 'simplernn_scaling_summary.csv'
if scaling_path.exists():
    scaling_df = pd.read_csv(scaling_path)
    print("SimpleRNN scaling summary:")
    display(scaling_df)
else:
    print(f'scaling summary not found at {scaling_path}')

rnn_path = artifacts_dir / 'torch_rnn_landmark_board_250000_with_history.json'
if rnn_path.exists():
    payload = json.loads(rnn_path.read_text(encoding='utf-8'))
    print("\n250k PyTorch SimpleRNN reported results:")
    print({k: payload[k] for k in ['test_accuracy', 'test_log_loss', 'train_games', 'test_games']})
else:
    print(f'250k RNN metrics not found at {rnn_path}')


ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Smoke test on a 60-game sample (~2-4 min on CPU). Outputs go to artifacts/colab_runs/.

import subprocess, sys

result = subprocess.run(
    [sys.executable, 'scripts/run_colab_repro.py', '--run-rnn-smoke', '--rnn-epochs', '2'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)


## **Experimental Results**

### Feature quality vs. model complexity

**What features the model sees matters more than which model family is used.** Adding board-state information (material counts, castling rights, legal move count) caused a much larger accuracy jump than switching to a more complex model.

Starting with the light feature set (Elo, clock, and move-shape signals only), logistic regression reached `45.7%` test accuracy with log loss `0.911`. Histogram gradient boosting on the same features reached `46.1%`, essentially the same. Both are close to the baseline of predicting the winner by Elo alone.

Switching to board-aware features while keeping logistic regression jumped test accuracy to `57.2%` on `1k` games and `61.1%` on `10k` games. That single change (richer features, same model) outperformed every light-feature model by a wide margin.

### Tree-based and SVM models

On board-aware features, **histogram gradient boosting underperformed logistic regression** in both accuracy and calibration: its log loss ballooned to `1.826` because it failed to model draws reliably. XGBoost at `10k` games (`59.8%`, log loss `0.844`) was better but still did not beat the logistic baseline.

The **calibrated linear SVM** was the most competitive alternative. At `10k` games it reached `61.0%` accuracy and log loss `0.795`, nearly matching board logistic regression, with an average landmark accuracy of `65.7%`.

### MLP models

MLP classifiers achieved similar overall accuracy to logistic regression (roughly `54–55%`) but with **much higher log loss** (3–5 range), showing poorly calibrated probabilities. Scaling to `10k` games helped only slightly. The draw class remained poorly predicted across all MLP variants. The MLP architecture did not suit this problem well at the scales tested.

### Recurrent models

Because chess is a sequence of moves, recurrent models have a structural advantage: they see the full game history up to the current position rather than a single snapshot. The basic `SimpleRNN(64)` on `1k` board-aware landmark sequences reached `59.8%` test accuracy and log loss `0.807`, already competitive with snapshot models. Scaling to `10k` improved it to `61.5%` with log loss `0.774`.

The focused Keras RNN sweep on `1k` games showed the best single variant to be a `GRU(64)` with dropout `0.2` and midgame loss weighting (moves `20–50` weighted by `1.5`), reaching `61.9%` accuracy and log loss `0.774` with the strongest average Brier score.

### Scaling the SimpleRNN

The shard-based PyTorch pipeline made it practical to push the basic `SimpleRNN` to the full `8.63M` filtered games. Performance improved consistently with scale:

| Dataset | Test Accuracy | Test Log Loss | Avg Landmark Accuracy | Avg Landmark Brier | Move 5 Acc | Move 10 Acc | Move 20 Acc | Move 30 Acc | Move 40 Acc |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 1k | 0.598 | 0.807 | 0.628 | 0.471 | 0.492 | 0.471 | 0.623 | 0.604 | 0.783 |
| 10k | 0.615 | 0.774 | 0.672 | 0.424 | 0.529 | 0.536 | 0.627 | 0.687 | 0.701 |
| 50k | 0.637 | 0.747 | 0.690 | 0.407 | 0.534 | 0.572 | 0.645 | 0.702 | 0.737 |
| 100k | 0.631 | 0.755 | 0.700 | 0.403 | 0.529 | 0.560 | 0.644 | 0.700 | 0.718 |
| 250k | 0.628 | 0.763 | 0.689 | 0.420 | 0.529 | 0.555 | 0.640 | 0.703 | 0.714 |
| 1M | 0.634 | 0.753 | 0.704 | 0.396 | 0.532 | 0.562 | 0.646 | 0.706 | 0.722 |
| 8.63M | 0.636 | 0.749 | 0.742 | 0.353 | 0.534 | 0.563 | 0.650 | 0.709 | 0.724 |

The full-dataset run produced the strongest held-out test accuracy (`63.6%`), the strongest average landmark accuracy (`74.2%`), and the best Brier score (`0.353`) of any experiment. Gains became more gradual at larger scales but did not plateau within the range tested.

### Overall model comparison

| Model | Dataset | Evaluation | Test Accuracy | Test Log Loss | Avg Landmark Accuracy | Avg Landmark Brier | Move 5 Acc | Move 10 Acc | Move 20 Acc | Move 30 Acc | Move 40 Acc |
|---|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| Light Logistic Regression | 1k | Snapshot | 0.457 | 0.911 | 0.441 | 0.579 | 0.472 | 0.487 | 0.437 | 0.453 | 0.383 |
| Light HistGradientBoosting | 1k | Snapshot | 0.461 | 2.221 | 0.440 | 0.899 | 0.451 | 0.465 | 0.457 | 0.443 | 0.433 |
| Board Logistic Regression | 1k | Snapshot | 0.572 | 0.837 | 0.607 | 0.492 | 0.467 | 0.535 | 0.576 | 0.594 | 0.700 |
| Board Logistic Regression | 10k | Snapshot | 0.611 | 0.795 | 0.665 | 0.455 | 0.525 | 0.559 | 0.639 | 0.694 | 0.683 |
| Board SVM (calibrated linear) | 1k | Snapshot | 0.578 | 0.818 | 0.606 | 0.493 | 0.472 | 0.535 | 0.570 | 0.585 | 0.717 |
| Board SVM (calibrated linear) | 10k | Snapshot | 0.610 | 0.795 | 0.657 | 0.458 | 0.527 | 0.559 | 0.642 | 0.693 | 0.679 |
| Board HistGradientBoosting | 1k | Snapshot | 0.539 | 1.826 | 0.559 | 0.722 | 0.492 | 0.471 | 0.570 | 0.576 | 0.583 |
| Board XGBoost | 10k | Snapshot | 0.598 | 0.844 | N/A | N/A | N/A | N/A | N/A | N/A | N/A |
| Board MLP (128,64) | 1k | Snapshot | 0.541 | 3.280 | 0.556 | 0.772 | 0.487 | 0.481 | 0.517 | 0.519 | 0.583 |
| Board Deep MLP (256,128,64) | 1k | Snapshot | 0.543 | 3.668 | 0.555 | 0.779 | 0.451 | 0.497 | 0.556 | 0.576 | 0.583 |
| Board Deep MLP (256,128,64) | 10k | Snapshot | 0.545 | 3.072 | N/A | N/A | N/A | N/A | N/A | N/A | N/A |
| Board Balanced MLP (256,128,64)+OS | 10k | Snapshot | 0.542 | 2.935 | N/A | N/A | N/A | N/A | N/A | N/A | N/A |
| Board Wide MLP (512,256,128) | 10k | Snapshot | 0.541 | 5.626 | N/A | N/A | N/A | N/A | N/A | N/A | N/A |
| Basic SimpleRNN | 1k | Landmark Sequence | 0.598 | 0.807 | 0.628 | 0.471 | 0.513 | 0.562 | 0.636 | 0.651 | 0.750 |
| Basic SimpleRNN | 10k | Landmark Sequence | 0.615 | 0.774 | 0.672 | 0.424 | 0.529 | 0.536 | 0.628 | 0.687 | 0.701 |
| Basic SimpleRNN | 250k | Landmark Sequence | 0.628 | 0.763 | 0.689 | 0.420 | 0.529 | 0.555 | 0.640 | 0.703 | 0.714 |
| Basic SimpleRNN | 8.63M | Landmark Sequence | 0.636 | 0.749 | 0.742 | 0.353 | 0.534 | 0.563 | 0.650 | 0.709 | 0.724 |
| Best Tuned GRU (dropout + midgame weighting) | 1k | Landmark Sequence | 0.619 | 0.774 | 0.637 | 0.442 | 0.528 | 0.540 | 0.642 | 0.585 | 0.783 |
| Best Tuned GRU (dropout + midgame weighting) | 10k | Landmark Sequence | 0.622 | 0.769 | 0.669 | 0.427 | 0.536 | 0.545 | 0.647 | 0.697 | 0.685 |

### Top 3 models at 50,000 games

To test how the strongest models scale, the extraction was expanded to `50k` games. The basic `SimpleRNN` scaled best; the tuned GRU regressed and fell below both the SimpleRNN and the logistic baseline, suggesting it was over-fitted to the smaller development setting.

| Model | Test Accuracy | Test Log Loss | Avg Landmark Accuracy | Avg Landmark Brier | Move 5 Acc | Move 10 Acc | Move 20 Acc | Move 30 Acc | Move 40 Acc |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| Board Logistic Regression (50k) | 0.618 | 0.783 | 0.673 | 0.436 | 0.522 | 0.550 | 0.631 | 0.688 | 0.732 |
| Basic SimpleRNN (50k) | 0.637 | 0.747 | 0.690 | 0.407 | 0.534 | 0.572 | 0.645 | 0.702 | 0.737 |
| Tuned GRU (50k) | 0.610 | 0.809 | 0.596 | 0.536 | 0.523 | 0.539 | 0.618 | 0.672 | 0.717 |

### RNN variant sweep on 1,000 games

> **Note on the sweep results:** The Test Accuracy and Test Log Loss shown here (`0.614` / `0.815` for the plain SimpleRNN) differ slightly from the `1k` row in the Overall Comparison table (`0.598` / `0.807`). Both use the same `1,000`-game board-aware dataset but from separate training runs with different random seeds. The sweep is reported as-is so the relative comparison between variants remains valid.

| RNN Variant | Key Settings | Test Accuracy | Test Log Loss | Avg Landmark Accuracy | Avg Landmark Brier | Move 5 Acc | Move 10 Acc | Move 20 Acc | Move 30 Acc | Move 40 Acc |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| SimpleRNN | `SimpleRNN(64)`, `lr=1e-3`, `batch=64` | 0.614 | 0.815 | 0.638 | 0.478 | 0.513 | 0.562 | 0.636 | 0.651 | 0.750 |
| GRU | `GRU(64)`, `lr=1e-3`, `batch=64` | 0.582 | 0.842 | 0.603 | 0.486 | N/A | N/A | N/A | N/A | N/A |
| LSTM | `LSTM(64)`, `lr=1e-3`, `batch=64` | 0.585 | 0.821 | 0.616 | 0.488 | N/A | N/A | N/A | N/A | N/A |
| Stacked GRU | `GRU(64,64)`, `lr=1e-3`, `batch=64` | 0.595 | 0.820 | 0.624 | 0.476 | N/A | N/A | N/A | N/A | N/A |
| GRU + Dropout | `GRU(64)`, `dropout=0.2`, `rec_dropout=0.1` | 0.613 | 0.787 | 0.633 | 0.468 | N/A | N/A | N/A | N/A | N/A |
| GRU Tuned LR | `GRU(64)`, `lr=5e-4`, `batch=128`, `epochs=40` | 0.586 | 0.832 | 0.610 | 0.490 | N/A | N/A | N/A | N/A | N/A |
| GRU Midgame Weighted | `GRU(64)`, weight moves `20-50` by `1.5` | 0.607 | 0.790 | 0.634 | 0.458 | N/A | N/A | N/A | N/A | N/A |
| GRU Draw Weighted | `GRU(64)`, draw weight `2.0` | 0.578 | 0.838 | 0.593 | 0.493 | N/A | N/A | N/A | N/A | N/A |
| GRU Dropout + Midgame Weighting | `GRU(64)`, `dropout=0.2`, `rec_dropout=0.1`, weight moves `20-50` by `1.5` | 0.619 | 0.774 | 0.637 | 0.442 | 0.528 | 0.540 | 0.642 | 0.585 | 0.783 |

### Landmark accuracy across game stages

> **Note for Colab:** The plot images below are generated by the local training scripts and are not tracked in the repository. They will not display in Colab unless regenerated. The underlying data is available in `artifacts/reported_results/`.

*(SimpleRNN 100,000-game loss history, generated locally, not available in Colab)*

*(SimpleRNN 250,000-game loss history, generated locally, not available in Colab)*

*(Top 3 models on 50k games: landmark accuracy, generated locally, not available in Colab)*

*(Basic SimpleRNN scaling comparison, generated locally, not available in Colab)*

*(Landmark accuracy comparison across all saved models, generated locally, not available in Colab)*

All board-aware and recurrent models show the same qualitative pattern: accuracy is lowest in the opening (where the position is most uncertain) and rises steadily through the middlegame and endgame. The light-feature models hover near `44%` average landmark accuracy regardless of game stage. The gap between light and board-aware features is visible from move `10` onward and widens as the game develops.

## **Conclusions**

**Feature engineering drove the largest gains.** Adding a compact board summary produced a much bigger improvement than any change in model architecture. Simple calibrated models (logistic regression, calibrated SVM) outperformed more complex ones (MLP, histogram gradient boosting) on both accuracy and probability calibration, well-calibrated predictions require matching the model to the data, not just increasing capacity.

Recurrent models provided the clearest structural advantage. They naturally incorporate the game's history and consistently outperformed snapshot models under landmark evaluation. The basic `SimpleRNN` scaled steadily from `1k` to `8.63M` games, finishing with `63.6%` test accuracy, `74.2%` average landmark accuracy, and a Brier score of `0.353`.

The landmark evaluation design was important for honest reporting. A single pooled accuracy number would obscure the fact that prediction is genuinely uncertain in the opening and becomes substantially stronger by moves `30–40`. The final model's progression from `53.4%` at move `5` to `72.4%` at move `40` is a more informative description of its behaviour than a single headline number.

Practical limitations remain. Full board-aware extraction over all `8.63M` games is computationally heavy; the shard-based GPU pipeline made it feasible but still required multiple hours of training. The draw class remained hard to predict for all models, and the recurrent models do not yet incorporate raw board positions or piece-square maps, which could provide richer spatial context. Both are clear directions for improvement.

## **References**
- Lichess Open Database. Official website and data source: https://database.lichess.org/ . Used as the raw PGN source for all experiments.
- `python-chess` documentation: https://python-chess.readthedocs.io/ . Used for PGN parsing and legal board reconstruction.
- scikit-learn documentation: https://scikit-learn.org/stable/ . Used for logistic regression, calibrated SVM, histogram gradient boosting, and MLP baselines.
- TensorFlow documentation: https://www.tensorflow.org/ and Keras documentation: https://keras.io/ . Used for the earlier recurrent-neural-network experiments.
- PyTorch documentation: https://pytorch.org/docs/stable/index.html . Used for the shard-based GPU SimpleRNN experiments at larger scale.
- XGBoost documentation: https://xgboost.readthedocs.io/ . Used for the later board-aware boosted-tree comparison.
- Course Project Guidelines document and Course Material
